# Train an HGT + TCN network with hierarchical skills and auxiliary intent

Run **26_prepare_skill_intent_target_relative_missile_nn_duel_skill_forecasting.ipynb**
first. This is the **temporal-convolution counterpart to notebook 27**: one fitted
network with seed 42, the same prepared data, HGT-v2 spatial encoder, hierarchical
family/direction heads, auxiliary intent head, loss and training recipe. Notebook
27's completed outputs remain the Transformer reference. No new data preparation
is needed when its notebook-26 v2 dataset is already available.

**TCN means Temporal Convolutional Network.** The temporal Transformer and its
learned position embedding are replaced by four width-96 residual blocks. Each
block has two causal Conv1d layers, kernel size 3, with block dilations **1, 2,
4, 8**, per-time channel LayerNorm, GELU and dropout 0.25. Left padding preserves
all 51 time positions; normalization never mixes future time positions into the
past. The receptive field is `1 + 2*(3-1)*(1+2+4+8) = 61` frames, covering the
complete 51-frame history. The final time position, followed by the existing final
LayerNorm, feeds all prediction heads. There is **no temporal self-attention** or
learned position embedding. The HGT's spatial relation attention is retained.
This is a different temporal architecture and parameter budget, not a claim of
better weighted F1 or equal inference cost.

The HGT remains two width-64 layers with four spatial heads, separately gated
relation families, 18 physical edge features and a learned threat prior. Both
aircraft accelerations, target-relative missile positions/velocities and the
observer's current skill are unchanged. Inputs are `(batch,51,115)` for the full
catalogue. Missile slots remain frame-local sets, not persistent tracks.

The readout predicts 29 families, left/right conditional on four mirrored families,
and three intent groups at +1/+2/+5 seconds. Joint probabilities reconstruct all
33 skills. Intent means the frozen offensive/defensive/context-dependent catalogue
groups, not an independently observed tactical intention. Future labels are never
encoder inputs. Exact-skill reports use the joint skill argmax; optional greedy
family-then-side decoding and true-family-conditioned direction reports retain
their separate definitions.

Training preserves notebook 27's actual recipe: family-first partial-credit loss,
equal family/side rewards, training-fitted inverse-square-root family and side
balancing, auxiliary intent CE weight 0.25, AdamW, L1, warm-up/cosine scheduling,
and batch size 1024. The family-weight ratio cap is **100** and conditional-side
cap **20**, matching the reviewed notebook-27 configuration. These are experimental
settings, not a claim that rare-class weighting optimizes weighted F1.

For a mirrored target, unweighted probability credit is
`C = (P(true_family) + P(true_family)*P(true_side|true_family))/2`.
Singleton credit is `P(true_family)`. Training applies the true side's fitted
weight only to the bounded side penalty, then the shared family weight to the
combined skill loss. Missing targets supervise no term. Hard family-first credit
is 1 for the correct family/side, 0.5 for the correct family/wrong side and zero
for a wrong family. Singleton correctness receives one. Final credit reports are
unweighted and remain distinct from exact-skill NLL and weighted F1.

Grouped partitions, training-only scaling, non-overlapping histories and masks
are unchanged. **Unweighted joint-skill test NLL still selects checkpoints**;
validation is evaluated only after training ends. This keeps the architecture
comparison's selection policy unchanged. The partition named test is a selection
set. One seed does not establish a reliable improvement; compare every horizon's
weighted F1 on matched examples and confirm promising results across seeds.

Artifacts are separate under `artifacts/nn_duel_skill_intent_tcn_forecasting/`,
with `comparisons/latest_hgt_hierarchical_intent_tcn_run.json`. Prepared data still
comes from `artifacts/nn_duel_skill_intent_forecasting/latest_dataset.json` or
`BVR_NN_INTENT_DATA`. Set `BVR_NN_INTENT_TCN_MLFLOW=0` to disable tracking. Run All
trains once, writes resumable checkpoints, prediction NPZs, CSVs and diagnostics.
CPU uses FP32; CUDA may train in BF16 with FP32 graph reductions. All final
metrics use FP32; the inherited math-attention setting concerns spatial HGT
compatibility and shared evaluation policy, not a temporal Transformer.

Start a fresh run for this TCN architecture. Notebook-27 checkpoints cannot resume
as TCNs. For a TCN resume/report-only session set the one `EXISTING_RUN` path;
source, config, dataset and taxonomy checks protect reuse. Every code cell is timed.


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction import skill_intent_forecasting as intent_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
DATA_ROOT = ROOT / "artifacts/nn_duel_skill_intent_forecasting"
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_skill_intent_tcn_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
from bvr_behavior_prediction.paired_skill_forecasting import (
    DEFAULT_TRAINING, train_model, final_evaluation, metric_tables, resolve_precision,
    fit_class_weighting, class_weight_table, fit_handedness_weighting, handedness_weight_table,
)
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts
from bvr_behavior_prediction.enemy_missile_diagnostics import save_diagnostics

DATA_OVERRIDE = os.environ.get("BVR_NN_INTENT_DATA")
if not DATA_OVERRIDE and not (DATA_ROOT / "latest_dataset.json").exists():
    raise FileNotFoundError("Run notebook 26 first, or set BVR_NN_INTENT_DATA to its dataset directory.")
receipt = None if DATA_OVERRIDE else read_json(DATA_ROOT / "latest_dataset.json")
DATA = Path(DATA_OVERRIDE or receipt["directory"]).expanduser().resolve()
if receipt and sha256(DATA / "manifest.json") != receipt["manifest_sha256"]:
    raise ValueError("The prepared dataset changed since notebook 26 completed.")
data = intent_data.IntentNNDuelData(DATA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42  # Exactly one training seed and one fitted network.
ARCHITECTURE = "hgt_hierarchical_intent_tcn"
TRAIN_MODEL = True
REEVALUATE_SAVED = False
RESUME = False
EXISTING_RUN = None  # Path required for resume or report-only execution.
if (RESUME or not TRAIN_MODEL) and not EXISTING_RUN:
    raise ValueError("Set EXISTING_RUN when resuming or reporting.")
if EXISTING_RUN and TRAIN_MODEL and not RESUME:
    raise ValueError("Use RESUME=True for an existing run, or clear EXISTING_RUN for a fresh fit.")
RUN_STAMP = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUN = (Path(EXISTING_RUN).expanduser().resolve() if EXISTING_RUN else
       EXPERIMENT_ROOT / "runs" / f"nn-duel-hierarchical-skill-intent-hgt-tcn-s{SEED}_{RUN_STAMP}")
CONFIG = {key: value for key, value in DEFAULT_TRAINING.items()
          if key not in {"heads", "layers"}}  # No temporal attention settings.
CONFIG.update(architecture=ARCHITECTURE, seed=SEED, intent_loss_weight=0.25,
              hierarchical_loss="partial_credit", family_reward=1.0, handedness_reward=1.0, observer_acceleration=True, target_relative_missiles=True,
              target_relative_missile_velocities=True, width=96, tcn_kernel_size=3, tcn_dilations=[1, 2, 4, 8],
              graph_width=64, graph_layers=2, graph_heads=4, hgt_version=2,
              threat_cpa_horizon_s=30.0, threat_time_scale_s=5.0, threat_prior_strength=1.0,
              missile_distance_floor_m=1000.0, missile_distance_power=1.0,
              dropout=0.25, epochs=100, learning_rate=3e-4,
              lr_schedule="warmup_cosine", warmup_fraction=0.05,
              warmup_start_ratio=0.1, min_lr_ratio=0.05,
              early_stopping=True, early_stopping_min_epochs=20,
              early_stopping_patience=10, l1_lambda=1e-6,
              class_weighting="family_sqrt_inverse", max_class_weight_ratio=100.0,
              handedness_weighting="sqrt_inverse", max_handedness_weight_ratio=20.0,
              overtraining_shutdown=True, overtraining_min_epochs=20, overtraining_patience=3,
              overtraining_relative_increase=0.05, overtraining_min_gap=0.1,
              precision="bf16", evaluation_precision="fp32", evaluation_attention="math",
              batch_size=1024,
              evaluation_batch_size=1024, epoch_streaming_metrics=True,
              evaluation_frame_cache=True, evaluation_frame_batch_size=8192,
              evaluation_cache_max_frames=65536, data_backend="prefetch" if DEVICE == "cuda" else "direct",
              num_workers=2, cpu_threads=4, compile_model=False)
MLFLOW_ENABLED = os.environ.get("BVR_NN_INTENT_TCN_MLFLOW", "1") == "1"
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI",
    f"sqlite:///{(EXPERIMENT_ROOT / 'mlflow.db').as_posix()}")
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": "nn-duel-skill-intent-tcn-forecasting", "log_per_class": True,
            "log_model": True,
            "artifact_root": (EXPERIMENT_ROOT / "mlflow_artifacts").as_uri()
                             if MLFLOW_TRACKING_URI.startswith("sqlite:") else None}
TCN_RECEPTIVE_FIELD = 1 + 2 * (CONFIG["tcn_kernel_size"] - 1) * sum(CONFIG["tcn_dilations"])
print("TCN blocks:", len(CONFIG["tcn_dilations"]), "| receptive field:", TCN_RECEPTIVE_FIELD, "frames")
print("Prepared dataset:", DATA)
print("Single model run:", RUN)
display(pd.Series({**CONFIG, "device": DEVICE,
    "effective_precision": resolve_precision(DEVICE, CONFIG["precision"]),
    "input_features": len(data.manifest["features"]), "classes": len(data.classes),
    "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))
display(pd.DataFrame(data.manifest["counts"]).T)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.synchronize()

# This audit reads only labels retained in the training partition.
CLASS_WEIGHTING = fit_class_weighting(data, CONFIG)
CLASS_WEIGHT_AUDIT = class_weight_table(CLASS_WEIGHTING)
with pd.option_context("display.max_rows", None):
    display(CLASS_WEIGHT_AUDIT.pivot(index="skill", columns="horizon_s",
                                    values=["training_support", "weight"]).round(4))
print("Zero-training-support skill/horizon pairs:")
display(CLASS_WEIGHT_AUDIT.loc[CLASS_WEIGHT_AUDIT.training_support.eq(0)])

print("Intent class order:", data.intent_classes)
print("Auxiliary intent loss weight:", CONFIG["intent_loss_weight"])
display(pd.read_csv(DATA / "intent_support_by_split.csv"))

print("Family count:", len(data.family_classes))
print("Conditional-direction families:", data.manifest["skill_hierarchy"]["paired_families"])
print("Direction order:", data.direction_classes)
display(pd.read_csv(DATA / "direction_support_by_split.csv"))

print("Skill objective:", CONFIG["hierarchical_loss"])
print("Family credit:", CONFIG["family_reward"], "| additional handedness bonus:", CONFIG["handedness_reward"])

# Counts are read only from retained training targets, separately for each horizon and pair.
HANDEDNESS_WEIGHTING = fit_handedness_weighting(data, CONFIG)
HANDEDNESS_WEIGHT_AUDIT = handedness_weight_table(HANDEDNESS_WEIGHTING) if HANDEDNESS_WEIGHTING else None
if HANDEDNESS_WEIGHT_AUDIT is not None:
    with pd.option_context("display.max_rows", None):
        display(HANDEDNESS_WEIGHT_AUDIT.pivot(index=["horizon_s", "family"], columns="direction",
                                            values=["training_support", "conditional_frequency", "weight"]).round(5))
    print("Unseen training sides (retained in the model):")
    display(HANDEDNESS_WEIGHT_AUDIT.loc[HANDEDNESS_WEIGHT_AUDIT.training_support.eq(0)])


## Fit one HGT + TCN network

The four causal residual blocks operate on the HGT's frame embeddings. Per-time
LayerNorm normalizes channels only, so representations do not use later frames.
The last time position can depend on the entire five-second input. No convolution
receives true future labels, and no temporal state is carried between flights or
batches. The final shared representation feeds the unchanged family, side and
intent heads.

A TCN shares local motion filters across time and can combine patterns with
increasing dilations. Its context is finite: increasing history requires a suitable
receptive field and a fresh compatible dataset/configuration. It cannot recover
information discarded by spatial pooling or unobserved opponent decisions. It is
not assumed faster at only 51 frames; measure complete training and inference.

The family and conditional-handedness weights displayed above are fitted only on
retained training labels, separately at each horizon. Family weights are shared
by both sides. Conditional side weights compensate for imbalance inside each
mirrored family, avoiding a second leaf-level correction. Unsupported targets
cannot be learned from weighting alone. Counts, mappings and fitted weights are
saved in JSON/CSV and checkpoints and must match on resume.

The data objective is the family-weighted sum of family CE and the handedness-
weighted partial-credit side penalty, plus `0.25 * intent_CE`. Each horizon averages
over all valid skill labels before represented horizons are averaged equally.
`1e-6 * sum(abs(trainable matrix/tensor weights))` is added as L1, excluding biases
and normalization scales; AdamW weight decay remains 0.01. These choices match
notebook 27, while the number and type of regularized parameters can differ.
History separates data loss, family/side contributions, intent NLL and L1.

Checkpoint selection and both stopping rules use unweighted original-skill test
NLL, not the balanced training objective or weighted F1. Checks begin after epoch
20; ten stale epochs trigger patience stopping. The independent overtraining rule
requires three consecutive epochs satisfying its configured NLL/gap criteria.
The selected checkpoint can precede either stopping minimum. The fixed warm-up/
cosine schedule is based on the planned optimizer updates, with state saved for
exact resume. Validation is not used for model selection.

Epoch evaluation retains notebook 27's bounded frame-local HGT cache and streaming
metric accumulation; neither changes the temporal TCN or crosses a weight update.
Final evaluation retains complete prediction arrays. FP32 evaluation and numerical
failure diagnostics remain active. Per-class reports are displayed every five
epochs and at training completion, while every epoch's reports are saved.

Exactly one seed is fitted. Changing TCN kernel/dilations, loss, weighting, batch
size or epoch budget requires a fresh run. Reuse completed reports only with
`TRAIN_MODEL=False` and `REEVALUATE_SAVED=False`.


In [ ]:
%%time
PER_CLASS_DISPLAY_EVERY = 5
last_displayed_per_class_epoch = None

def display_per_class(per_class, epoch):
    global last_displayed_per_class_epoch
    print(f"Per-class results at epoch {epoch}:")
    with pd.option_context("display.max_rows", None):
        display(per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                 values=["class_nll", "f1", "support"]).round(4))
    last_displayed_per_class_epoch = int(epoch)

def epoch_output(row, summary, per_class):
    print(f"Epoch {row['epoch']}: train NLL={row['train_nll']:.5f}, "
          f"test NLL={row['test_nll']:.5f}, "
          f"train data loss={row['train_data_loss']:.5f}, "
          f"intent NLL={row['train_intent_nll']:.5f}, "
          f"family NLL={row['train_family_nll']:.5f}, "
          f"balanced side loss={row['train_handedness_data_loss']:.5f}, "
          f"test credit={row['test_hierarchical_credit']:.3f}, "
          f"L1={row['train_l1_penalty']:.5f}, "
          f"gap={row['generalization_gap']:.5f}, "
          f"LR={row['learning_rate']:.3g}, "
          f"overtraining streak={row['overtraining_epochs']}, {row['seconds']:.1f}s", flush=True)
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    if row["epoch"] % PER_CLASS_DISPLAY_EVERY == 0 or row["early_stopped"] or row["epoch"] == CONFIG["epochs"]:
        display_per_class(per_class, row["epoch"])

tracking = {**TRACKING, "run_name": RUN.name}
if TRAIN_MODEL:
    history = train_model(data, RUN, CONFIG, callback=epoch_output, device=DEVICE,
                          resume=RESUME, tracking=tracking)
else:
    history = pd.read_csv(RUN / "history.csv")
status = read_json(RUN / "training_status.json")
if status["status"] not in {"completed", "early_stopped"}:
    raise ValueError("Training has not concluded.")
saved_run = read_json(RUN / "run.json")
if saved_run["data_manifest_sha256"] != sha256(DATA / "manifest.json"):
    raise ValueError("Saved run belongs to another prepared dataset.")
if (saved_run["classes"] != data.classes or saved_run["feature_names"] != data.manifest["features"]
        or saved_run["intent_supervision"] != data.manifest["intent_supervision"]
        or saved_run["skill_hierarchy"] != data.manifest["skill_hierarchy"]):
    raise ValueError("Saved catalogue, feature order or intent taxonomy differs.")
if saved_run.get("handedness_weighting") != HANDEDNESS_WEIGHTING:
    raise ValueError("Saved training-fitted handedness weights differ.")
if saved_run["training"] != CONFIG:
    raise ValueError("Requested configuration differs from the saved run.")
display(history.tail())
# Also show the final table for completed resumes and report-only execution.
final_epoch = int(history.iloc[-1]["epoch"])
if last_displayed_per_class_epoch != final_epoch:
    display_per_class(pd.read_csv(RUN / "epochs" / f"epoch_{final_epoch:03d}_per_class.csv"), final_epoch)
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Final evaluation and persisted reports

Training must finish before this cell accesses validation. Completed skill and
intent predictions are reused unless `REEVALUATE_SAVED=True`. No additional seed
is fitted. The completed-run receipt is written only after all reports finish.


In [ ]:
%%time
required_finals = ["final_metrics.csv", "final_per_class.csv", "selection.json",
                   "final_intent_metrics.csv", "final_intent_per_class.csv",
                   "final_family_metrics.csv", "final_family_per_class.csv",
                   "final_direction_metrics.csv", "final_direction_per_class.csv", "final_hierarchical_credit_metrics.csv",
                   *[f"final_family_{split}.npz" for split in SPLITS],
                   *[f"final_direction_{split}.npz" for split in SPLITS],
                   *[f"final_{split}.npz" for split in SPLITS],
                   *[f"final_intent_{split}.npz" for split in SPLITS]]
if not REEVALUATE_SAVED and all((RUN / name).is_file() for name in required_finals):
    best_epoch = read_json(RUN / "selection.json")["epoch"]
    summary = pd.read_csv(RUN / "final_metrics.csv")
    print("Reusing saved final predictions:", RUN)
else:
    best_epoch, summary, scores, results = final_evaluation(data, RUN, DEVICE, tracking=tracking)
    del results
display(Markdown(f"**Single seed {SEED}, skill-test-selected epoch {best_epoch}**"))
display(summary.round(4))
display(pd.read_csv(RUN / "final_intent_metrics.csv").round(4))
display(pd.read_csv(RUN / "final_family_metrics.csv").round(4))
print("Normalized family credit plus handedness bonus:")
display(pd.read_csv(RUN / "final_hierarchical_credit_metrics.csv").round(4))
print("Direction metrics condition on the true mirrored family, excluding singletons:")
display(pd.read_csv(RUN / "final_direction_metrics.csv").round(4))

# Both baselines use only retained training labels; no held-out labels fit probabilities.
for task, labels, classes in (("skill", data.targets, data.classes),
                               ("intent", data.intent_targets, data.intent_classes)):
    frequency = np.zeros((len(HORIZONS), len(classes)), np.float32)
    for h in range(len(HORIZONS)):
        selected = labels[data.indices["train"], h]
        counts = np.bincount(selected[selected >= 0], minlength=len(classes))
        frequency[h] = (counts + 1) / (counts.sum() + len(classes))
    rows = []
    for split in SPLITS:
        targets = labels[data.indices[split]]
        probabilities = np.broadcast_to(frequency, (len(targets), *frequency.shape))
        baseline, _, _ = metric_tables(targets, probabilities, classes, split)
        rows.append(baseline)
    baseline = pd.concat(rows, ignore_index=True)
    filename = "training_frequency_baseline.csv" if task == "skill" else "training_frequency_intent_baseline.csv"
    baseline.to_csv(RUN / filename, index=False)
    display(baseline.assign(task=task, model="Training frequency").round(4))

save_diagnostics(RUN, data=data)
intent_comparison = intent_data.save_intent_diagnostics(RUN, data)
display(intent_comparison.round(4))
if MLFLOW_ENABLED:
    log_run_artifacts(RUN)
receipt_path = EXPERIMENT_ROOT / "comparisons/latest_hgt_hierarchical_intent_tcn_run.json"
receipt_path.parent.mkdir(parents=True, exist_ok=True)
write_json(receipt_path, {
    "architecture": ARCHITECTURE, "run": str(RUN), "seed": SEED,
    "data_manifest_sha256": sha256(DATA / "manifest.json"),
    "configuration": CONFIG, "intent_supervision": data.manifest["intent_supervision"],
    "skill_hierarchy": data.manifest["skill_hierarchy"],
    "hierarchical_objective": saved_run["hierarchical_objective"],
    "handedness_weighting": saved_run["handedness_weighting"],
})
print("Trained checkpoint:", RUN / "best.pt")
print("Completed single-run receipt:", receipt_path)


## Inspect the single run

`diagnostics/` contains skill reports. `intent_diagnostics/` compares the intent head
and grouped skill probabilities, including calibration, per-class tables and
confusion plots for every partition. `final_intent_*.npz` retain exact aligned
sample IDs, targets, probabilities and per-example logit cross-entropies.

`final_family_*.npz` and `final_family_metrics.csv` report the family task.
`final_direction_*.npz` and `final_direction_metrics.csv` report direction **given
the true paired family**, masking all singletons and missing labels. Empty
horizons retain count zero and undefined NLL/accuracy. The primary 33-class reports
use joint skill probabilities and include errors from either decision level.

`final_hierarchical_credit_metrics.csv` reports family accuracy, realized normalized
family-first credit, continuous expected credit, and unweighted partial-credit
log loss per split/horizon. With the default rewards, correct family/wrong side
gets half credit, while correct family/correct side gets full credit.

`handedness_weights.csv` audits L/R support and fitted weights for every pair/horizon;
`class_weights.csv` shows shared family weights. The reward and joint-NLL reports
remain unweighted. Training objective logs apply the frozen balancing weights;
these are not used to alter predicted probabilities at inference.


In [ ]:
%%time
from IPython.display import Image

display(pd.read_csv(RUN / "intent_diagnostics/summary.csv").round(4))
display(Image(filename=str(RUN / "diagnostics/learning_curves.png")))
display(Image(filename=str(RUN / "intent_diagnostics/confusion_intent_head_validation.png")))
print("Network weights:", RUN / "best.pt")


## Use the hierarchical network

`best.pt` contains the HGT/causal-TCN encoder, family head, conditional direction
head, auxiliary intent head, normalization and frozen mappings. Reconstruct with
`make_model(len(saved["classes"]), saved["model_config"], saved["mean"], saved["std"])`,
load `saved["model_state"]` and call `eval()`.

For a normalized `(batch,51,115)` history:

```python
with torch.inference_mode():
    outputs = model(x, return_hierarchy=True)
    family_probabilities = outputs["family_logits"].softmax(-1)       # (batch,3,29)
    direction_probabilities = outputs["direction_logits"].softmax(-1) # (batch,3,4,2)
    skill_probabilities = outputs["skill_log_probs"].exp()           # (batch,3,33)
    joint_skill = skill_probabilities.argmax(-1)
    greedy = model.decode_hierarchy(outputs["family_logits"], outputs["direction_logits"])
```

`joint_skill` is used by the reported original-skill metrics. `greedy` contains
family-first `family_indices`, then `direction_indices` (-1 for a singleton),
and the corresponding original `skill_indices`. These decisions can differ when
a high-probability mirrored family splits its probability between left and right.

The class/family/paired-family/side order is in `saved["skill_hierarchy"]` and the
model configuration. Side order is **left, right**. Intent order is in
`saved["intent_supervision"]["classes"]`. Inputs still contain only the observer's
current skill; no future labels or predicted opponent family are encoder inputs.

The default `model(x)` / MLflow export returns normalized joint original-skill log
probabilities. Softmax remains compatible with the old skill-score interface.
Use the checkpoint factory and `return_hierarchy=True` to access every head.
`return_intent=True` returns the joint skill log probabilities and intent logits.
Training/resume requires the pinned base dataset and v2 label overlay; inference
from checkpoint weights does not read dataset files.

`model_config` records architecture `hgt_hierarchical_intent_tcn`, temporal width,
kernel size and dilation list, plus the HGT and head settings. The TCN checkpoints
have no temporal Transformer weights or learned position embeddings. Reloading
uses the shared factory; no notebook-only monkeypatch or custom class definition
is required. The inherited graph-frame cache remains valid because only the
frame-local spatial encoder is cached, never temporal outputs.
